In [ ]:
import glob
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("../Results/link_prediction")
N_VALUES = [50, 100, 150, 200]

for n in N_VALUES:
    pattern = RESULTS_DIR / f"lp_n{n}_r3_task*.csv"
    files = sorted(glob.glob(str(pattern)))

    if not files:
        print(f"[WARN] No files found for n={n}")
        continue

    print(f"Merging {len(files)} files for n={n}")

    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    out_file = RESULTS_DIR / f"lp_n{n}_r3_merged.csv"
    df.to_csv(out_file, index=False)

    print(f"   wrote {out_file}")


In [ ]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------
# Paths
# -------------------------
RESULTS_DIR = Path("../Results/link_prediction")
OUT_CSV = RESULTS_DIR / "linkpred_summary.csv"
OUT_TEX = RESULTS_DIR / "linkpred_summary.tex"

files = sorted(glob.glob(str(RESULTS_DIR / "lp_n*_task*.csv")))
if not files:
    raise FileNotFoundError("No link-prediction CSV files found")

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# -------------------------
# Metrics to aggregate
# -------------------------
metrics = {
    "ridge_np": ["ridge_np_roc_auc", "ridge_np_f1_best", "ridge_np_ece"],
    "mle_np":   ["mle_np_roc_auc",   "mle_np_f1_best",   "mle_np_ece"],
    "loc":      ["loc_roc_auc",      "loc_f1_best",      "loc_ece"],
    "cen":      ["cen_roc_auc",      "cen_f1_best",      "cen_ece"],
}

# -------------------------
# Group by (n, eps)
# -------------------------
agg_dict = {}
for method, cols in metrics.items():
    for c in cols:
        agg_dict[c] = "mean"

summary = (
    df.groupby(["n", "eps"])
      .agg(agg_dict)
      .reset_index()
      .sort_values(["n", "eps"])
)

summary.to_csv(OUT_CSV, index=False)
print(f"Wrote summary CSV to {OUT_CSV}")


In [ ]:
with open(OUT_TEX, "w") as f:
    for n_val, sdf in summary.groupby("n"):
        f.write("\\begin{table}[t]\n")
        f.write("\\centering\n")
        f.write(f"\\caption{{Link prediction performance ($n={int(n_val)}$)}}\n")
        f.write(f"\\label{{tab:lp_n{int(n_val)}}}\n")
        f.write("\\begin{tabular}{cccccccccc}\n")
        f.write("\\toprule\n")
        f.write(
            " $\\varepsilon$ & "
            "Ridge AUC & Ridge F1 & Ridge ECE & "
            "MLE AUC & MLE F1 & MLE ECE & "
            "Local AUC & Local F1 & Local ECE & "
            "Central AUC & Central F1 & Central ECE \\\\\n"
        )
        f.write("\\midrule\n")

        for _, r in sdf.iterrows():
            f.write(
                f"{r.eps:.3g} & "
                f"{r.ridge_np_roc_auc:.3f} & {r.ridge_np_f1_best:.3f} & {r.ridge_np_ece:.3f} & "
                f"{r.mle_np_roc_auc:.3f} & {r.mle_np_f1_best:.3f} & {r.mle_np_ece:.3f} & "
                f"{r.loc_roc_auc:.3f} & {r.loc_f1_best:.3f} & {r.loc_ece:.3f} & "
                f"{r.cen_roc_auc:.3f} & {r.cen_f1_best:.3f} & {r.cen_ece:.3f} \\\\\n"
            )

        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write("\\end{table}\n\n")

print(f"Wrote LaTeX table to {OUT_TEX}")


In [ ]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("../Results/link_prediction")
OUT_CSV = RESULTS_DIR / "linkpred_privacy_cost.csv"
OUT_TEX = RESULTS_DIR / "linkpred_privacy_cost.tex"

files = sorted(glob.glob(str(RESULTS_DIR / "lp_n*_task*.csv")))
if not files:
    raise FileNotFoundError(f"No task CSVs found in {RESULTS_DIR} matching lp_n*_task*.csv")

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# --- required columns
needed = {
    "n", "eps",
    "ridge_np_roc_auc", "ridge_np_f1_best", "ridge_np_ece",
    "mle_np_roc_auc",   "mle_np_f1_best",   "mle_np_ece",
    "loc_roc_auc",      "loc_f1_best",      "loc_ece",
    "cen_roc_auc",      "cen_f1_best",      "cen_ece",
}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# --- average over reps within each (n, eps)
cols_to_mean = sorted(list(needed - {"n", "eps"}))
summary = (
    df.groupby(["n", "eps"])[cols_to_mean]
      .mean()
      .reset_index()
      .sort_values(["n", "eps"])
)

# --- compute privacy-cost deltas
summary["d_loc_auc"] = summary["loc_roc_auc"] - summary["ridge_np_roc_auc"]
summary["d_loc_f1"]  = summary["loc_f1_best"] - summary["ridge_np_f1_best"]
summary["d_loc_ece"] = summary["loc_ece"]      - summary["ridge_np_ece"]

summary["d_cen_auc"] = summary["cen_roc_auc"] - summary["mle_np_roc_auc"]
summary["d_cen_f1"]  = summary["cen_f1_best"] - summary["mle_np_f1_best"]
summary["d_cen_ece"] = summary["cen_ece"]      - summary["mle_np_ece"]

# --- keep only what you want in the final CSV/table
out = summary[[
    "n", "eps",
    "d_loc_auc", "d_loc_f1", "d_loc_ece",
    "d_cen_auc", "d_cen_f1", "d_cen_ece",
]].copy()

out.to_csv(OUT_CSV, index=False)
print(f"Wrote CSV: {OUT_CSV}")

# -------------------------
# LaTeX (one big table)
# -------------------------
def fmt_eps(x):
    # compact scientific formatting
    return f"{x:.3g}"

def fmt(x):
    # 3 decimals, keep sign
    return f"{x:+.3f}"

with open(OUT_TEX, "w") as f:
    f.write("\\begin{table}[t]\n")
    f.write("\\centering\n")
    f.write("\\caption{Privacy cost in link prediction (differences vs non-private baselines)}\n")
    f.write("\\label{tab:lp_privacy_cost}\n")
    f.write("\\small\n")
    f.write("\\setlength{\\tabcolsep}{4pt}\n")
    f.write("\\begin{tabular}{ccccc|ccc}\n")
    f.write("\\toprule\n")
    f.write(
        "$n$ & $\\varepsilon$ & "
        "$\\Delta$AUC$_\\mathrm{loc}$ & $\\Delta$F1$_\\mathrm{loc}$ & $\\Delta$ECE$_\\mathrm{loc}$ & "
        "$\\Delta$AUC$_\\mathrm{cen}$ & $\\Delta$F1$_\\mathrm{cen}$ & $\\Delta$ECE$_\\mathrm{cen}$ \\\\\n"
    )
    f.write("\\midrule\n")

    last_n = None
    for _, r in out.iterrows():
        n_val = int(r["n"])
        if last_n is not None and n_val != last_n:
            f.write("\\midrule\n")  # visual separation between n-blocks
        last_n = n_val

        f.write(
            f"{n_val} & {fmt_eps(r['eps'])} & "
            f"{fmt(r['d_loc_auc'])} & {fmt(r['d_loc_f1'])} & {fmt(r['d_loc_ece'])} & "
            f"{fmt(r['d_cen_auc'])} & {fmt(r['d_cen_f1'])} & {fmt(r['d_cen_ece'])} \\\\\n"
        )

    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")

print(f"Wrote LaTeX: {OUT_TEX}")
